## Introduction
Cancer treatment is not equally effective for every patient. Tumours that look similar clinically can respond very differently to the same anticancer drug. One reason for this is the variety of genomic alterations present in the cells. Some of these alterations affect the biological pathways a drug targets and/or the mechanisms by which a cancer cell can survive drug treatment. If we can characterize the molecular features of a cancer cell, perhaps we can predit which drugs are likely to work on that cell. The Genomics of Drug Sensitivity in Cancer (GDSC) project has generated a large dataset combining:

- cancer cell lines
- genomic/molecular characteristics of those cell lines
- measurement of their responses to anticancer compounds

This makes the GDSC well suited to investigating the relationship between genomic features and drug response. 

Because the genomic feature space is enormous, Machine Learning (ML) provides methods for efficiently learning the relationships between genomic features and drug response. Importantly, it is uncertain whether genomic information actually contains enough signal to predict drug response.

**Question:** Can anticancer drug response be predicted from genomic features using machine-learning models trained on GDSC data, and which genomic features contribute most to predictive performance?

This question can be broken down into sevearal sub-questions:
1. **Predictability:** Is anticancer drug response predictable from genomic features?
2. **Model Performance:** Which machine-learning models provice the most useful predictive performance? Which models are most efficient? Is there an intersection between performance and efficiency?
3. **Generalization:** Does the predictive performance of the models generalize to unseen data?
4. **Biological Interpretation:** Which genomic features contribute most to predictive performance? Can we interpret the models to understand the biological mechanisms underlying drug response?
5. **Drug Specificity:** Are there specific drugs for which genomic features are particularly predictive of response? Conversely, are there drugs for which genomic features provide little predictive power?

**Null Hypothesis:** Genomic features do not contain enough information to predict anticancer drug response, with the methods explored.

**Alternative Hypothesis:** Genomic features exist that can provide predictive information about anticancer drug response, and the machine-learning models exploured can be trained to leverage this information effectively.

## Data
This analysis uses the official Wellcome Sanger Institute CancerRxGene GDSC release 8.4, dated 24 July 2022. It combines GDSC1 and GDSC2 fitted single-agent drug-response records with cell-line metadata from `Cell_Lines_Details.xlsx`.

Each observation represents one fitted response record for a cell-line/drug measurement. The loaded data retain the response variables, drug identifiers, COSMIC cell-line identifiers, tissue-of-origin descriptors, and cancer-type metadata. The complete release is inspected here before any cancer-type selection or preprocessing.

In [15]:
from gdsc.data import prepare_gdsc

gdsc = prepare_gdsc("data/raw")

gdsc.head()

/home/ajharris/Projects/gdsc-project/venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/home/ajharris/Projects/gdsc-project/venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,DATASET,NLME_RESULT_ID,NLME_CURVE_ID,COSMIC_ID,CELL_LINE_NAME,SANGER_MODEL_ID,TCGA_DESC,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,...,Copy Number Alterations (CNA),Gene Expression,Methylation,Drug\nResponse,TISSUE_OF_ORIGIN,TISSUE_DESCRIPTOR_2,CANCER_TYPE,Microsatellite \ninstability Status (MSI),Screen Medium,Growth Properties
0,GDSC1,361,17635802,684057,ES5,SIDM00263,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent
1,GDSC1,361,17636176,684059,ES7,SIDM00269,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent
2,GDSC1,361,17636568,684062,EW-11,SIDM00203,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent
3,GDSC1,361,17636912,684072,SK-ES-1,SIDM01111,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Semi-Adherent
4,GDSC1,361,17637300,687448,COLO-829,SIDM00909,SKCM,1,Erlotinib,EGFR,...,Y,Y,Y,Y,skin,melanoma,SKCM,MSS/MSI-L,R,Adherent


### Initial dataset characterization

The following summaries describe the loaded release before any preprocessing or analytical filtering. Tissue and cancer labels are retained as metadata; no cancer type is selected at this stage.

In [16]:
import pandas as pd

summary = pd.Series(
    {
        "observations": len(gdsc),
        "columns": len(gdsc.columns),
        "cell_lines": gdsc["COSMIC_ID"].nunique(),
        "drugs_or_compounds": gdsc["DRUG_NAME"].nunique(),
        "drug_identifiers": gdsc["DRUG_ID"].nunique(),
        "tissues": gdsc["TISSUE_OF_ORIGIN"].nunique(),
    },
    name="value",
)
summary

observations          575197
columns                   32
cell_lines               978
drugs_or_compounds       542
drug_identifiers         621
tissues                   19
Name: value, dtype: int64

In [17]:
tissue_summary = (
    gdsc.groupby("TISSUE_OF_ORIGIN")
    .agg(
        observations=("COSMIC_ID", "size"),
        cell_lines=("COSMIC_ID", "nunique"),
        drugs=("DRUG_ID", "nunique"),
    )
    .sort_values("observations", ascending=False)
)
tissue_summary

,observations,cell_lines,drugs
TISSUE_OF_ORIGIN,,,
lung_NSCLC,64499,108,621
urogenital_system,61288,104,621
leukemia,50008,84,621
aero_dig_tract,45354,77,621
lymphoma,40948,69,621
lung_SCLC,33558,63,621
skin,33253,58,621
nervous_system,32794,55,621
breast,31021,52,621


In [18]:
drug_summary = (
    gdsc.groupby("DRUG_NAME")
    .agg(
        observations=("COSMIC_ID", "size"),
        cell_lines=("COSMIC_ID", "nunique"),
        tissues=("TISSUE_OF_ORIGIN", "nunique"),
    )
    .sort_values("observations", ascending=False)
)

response_summary = gdsc[["AUC", "LN_IC50"]].agg(["count", "mean", "std", "min", "max"]).T
missingness = gdsc[[
    "COSMIC_ID", "CELL_LINE_NAME", "TISSUE_OF_ORIGIN", "CANCER_TYPE",
    "DRUG_ID", "DRUG_NAME", "AUC", "LN_IC50",
]].isna().sum().rename("missing")

print("Drug coverage:")
display(drug_summary.head(10))
print("Response distributions:")
display(response_summary)
print("Relevant-field missingness:")
display(missingness)


Drug coverage:


,observations,cell_lines,tissues
DRUG_NAME,,,
Selumetinib,3452,975,19
AZD7762,2812,975,19
SN-38,2804,975,19
PLX-4720,2804,975,19
Afatinib,2798,975,19
Avagacestat,2795,976,19
Olaparib,2794,975,19
AZD4547,2756,975,19
Pictilisib,2755,975,19


Response distributions:


,count,mean,std,min,max
AUC,575197.0,0.854714,0.177361,0.005996,0.999993
LN_IC50,575197.0,2.392694,2.691065,-10.577744,13.847363


Relevant-field missingness:


COSMIC_ID                0
CELL_LINE_NAME           0
TISSUE_OF_ORIGIN         0
CANCER_TYPE         102031
DRUG_ID                  0
DRUG_NAME                0
AUC                      0
LN_IC50                  0
Name: missing, dtype: int64

### Genomic availability and quality checks

The response files contain no genomic feature matrix. `Cell_Lines_Details.xlsx` records whether WES, CNA, gene expression, and methylation assays are available for each model; the actual feature files and their identifiers must be selected and joined in a later research-design step. The checks below report raw-data limitations without removing observations.

In [19]:
modality_flags = [
    "Whole Exome Sequencing (WES)",
    "Copy Number Alterations (CNA)",
    "Gene Expression",
    "Methylation",
]
modality_availability = gdsc[modality_flags].apply(lambda column: column.eq("Y").mean())

quality_checks = pd.Series(
    {
        "duplicate_curve_ids": gdsc.duplicated(["DATASET", "NLME_CURVE_ID"]).sum(),
        "missing_cell_line_ids": gdsc["COSMIC_ID"].isna().sum(),
        "missing_cell_line_names": gdsc["CELL_LINE_NAME"].isna().sum(),
        "missing_tissue_labels": gdsc["TISSUE_OF_ORIGIN"].isna().sum(),
        "missing_drug_ids": gdsc["DRUG_ID"].isna().sum(),
        "missing_auc": gdsc["AUC"].isna().sum(),
        "missing_ln_ic50": gdsc["LN_IC50"].isna().sum(),
        "auc_outside_0_1": ((gdsc["AUC"] < 0) | (gdsc["AUC"] > 1)).sum(),
    },
    name="count",
)

print("Cell-line assay availability flags (fraction marked Y):")
display(modality_availability)
print("Quality checks:")
display(quality_checks)

Cell-line assay availability flags (fraction marked Y):


Whole Exome Sequencing (WES)     1.000000
Copy Number Alterations (CNA)    0.995035
Gene Expression                  0.978428
Methylation                      0.965155
dtype: float64

Quality checks:


duplicate_curve_ids        0
missing_cell_line_ids      0
missing_cell_line_names    0
missing_tissue_labels      0
missing_drug_ids           0
missing_auc                0
missing_ln_ic50            0
auc_outside_0_1            0
Name: count, dtype: int64

### Data-to-Preprocessing checkpoint

This analysis uses official Sanger CancerRxGene GDSC release 8.4 (24 July 2022), combining GDSC1 and GDSC2 fitted single-agent response files with `Cell_Lines_Details.xlsx`. Each observation represents a cell-line/drug response record and retains COSMIC model ID, tissue of origin, and cancer metadata. AUC and LN_IC50 are available and complete in the loaded release; neither is selected as the final target yet.

The release metadata reports WES, CNA, gene-expression, and methylation availability flags, but does not include the genomic feature matrices themselves. Cancer type is missing for a substantial subset of observations, while tissue labels are retained. Before preprocessing, the project still needs to choose a genomic modality and source, define eligibility and missingness rules, select a response measure, and decide how cancer types will be compared. No observations have been imputed, filtered, normalized, or split at this checkpoint.

## Data Preprocessing

## Exploratory Analysis

## Baseline Models

## Machine-Learning Models

## Evaluation

## Feature Interpretation

## Results

## Discussion

## Reproducability